In [25]:
import os, json, requests, time
from dotenv import load_dotenv
from pageindex import PageIndexClient
from langchain_anthropic import ChatAnthropic
load_dotenv()
from anthropic import Anthropic



os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["PAGEINDEX_API_KEY"] = os.getenv("PAGEINDEX_API_KEY")
pi_client     = PageIndexClient(api_key=os.getenv("PAGEINDEX_API_KEY"))
anthropic_client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

In [8]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "./data/Course_Master_and_Serial_User_Manual.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: ./data/Course_Master_and_Serial_User_Manual.pdf
✅ Uploaded!
📋 Document ID: pi-cmvjwd5yn001q0dp88z6aewvw
   (Save this ID — you'll use it throughout the notebook)


In [10]:

# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("Building tree index")
print("This runs once per document - the index is cached for reuse")

while True:
    status_result = pi_client.get_document(doc_id)
    status=status_result.get("status")
    print(f"Status: {status}")
    if status == "completed":
        print("✅ Document processing complete!")
        break
    elif status == "failed":
        print("❌ Document processing failed!")
        break

    time.sleep(5)  # Wait 5 seconds before polling again
    

Building tree index
This runs once per document - the index is cached for reuse
Status: completed
✅ Document processing complete!


In [11]:
tree_result = pi_client.get_tree(doc_id,node_summary=True)
pageindex_tree = tree_result.get("result",[])

print(f"Tree index built with {len(pageindex_tree)} nodes")
print("\n Raw tree index:")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Tree index built with 1 nodes

 Raw tree index:
{
  "title": "LMS USER MANUAL",
  "node_id": "0000",
  "start_index": 1,
  "end_index": 25,
  "summary": "This section of the LMS User Manual for the Military College of Material Management provides a consolidated guide covering the system's core modules and functionalities. It outlines the table of contents and introduces the platform's structure, followed by detailed procedures for:\n- **Course Data Hub:** Creating, structuring, configuring, approving, and versioning foundational Course Masters (phases, faculty, syllabus, periods, and approval workflows).\n- **Course Serial:** Managing individual course runs based on the Annual Training Plan, including scheduling, faculty assignment, assessment marking schemes, physical standards, and approvals.\n- **Content Library:** Uploading, organizing, searching, filtering, and tracking the availability of instructional content for trainees.\n- **Trainee Management:** Enrolling, managing credentia

In [12]:

# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] LMS USER MANUAL  (p.?)
  └─ [0001] LMS USER MANUAL (intro)  (p.?)
  └─ [0002] Course Data Hub  (p.?)
    └─ [0003] Course Data Hub (intro)  (p.?)
    └─ [0004] Why is Course Master Created First?  (p.?)
    └─ [0005] Getting Started with Course Data Creation  (p.?)
    └─ [0006] How to Add Syllabus?  (p.?)
    └─ [0007] How to Add Periods?  (p.?)
    └─ [0008] How to Create Approval Rule?  (p.?)
    └─ [0009] Final Step  (p.?)
    └─ [0010] Why Create a New Version of a Course?  (p.?)
    └─ [0011] How to Create a New Version?  (p.?)
  └─ [0012] Course Serial - Introduction  (p.?)
    └─ [0013] Course Serial - Introduction (intro)  (p.?)
    └─ [0014] Getting Started with Course Serial Creation  (p.?)
      └─ [0015] Getting Started with Course Serial Creation (intro)  (p.?)
      └─ [0016] Step 4B – Confirm Updated Details  (p.?)
    └─ [0017] Creating the Other Assessment Marking Scheme  (p.?)
    └─ [0018] Finalizing and Submitting for Approval  (p

In [13]:

# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 30
   Each node = one retrievable section of the document


In [29]:
import os
import json
import re
from anthropic import Anthropic

# Initialize Anthropic client
anthropic_client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)


def llm_tree_search(
    query: str,
    tree: list,
    model: str = "claude-sonnet-4-6"
) -> dict:
    """
    PageIndex retrieval using Claude.

    Args:
        query: User's question.
        tree: Document tree structure.
        model: Anthropic Claude model name.

    Returns:
        Dictionary containing:
        - thinking: Brief explanation of the selected nodes.
        - node_list: List of relevant node IDs.
    """

    # 1. Compress the document tree
    def compress(nodes):
        out = []

        for node in nodes:
            entry = {
                "node_id": node.get("node_id", ""),
                "title": node.get("title", ""),
                "page": node.get("page_index", "?"),
                "summary": str(node.get("text", ""))[:150]
            }

            children = node.get("nodes", [])

            if children:
                entry["children"] = compress(children)

            out.append(entry)

        return out

    compressed_tree = compress(tree)

    # 2. Create the prompt
    prompt = f"""
You are a document retrieval assistant using a document tree.

Your task is to identify the node IDs most relevant to answering
the user's question.

Question:
{query}

Document Tree:
{json.dumps(compressed_tree, indent=2, ensure_ascii=False)}

Instructions:
- Select only node IDs present in the document tree.
- Select the most relevant nodes.
- If no node is relevant, return an empty list.
- Give a brief explanation of your selection.
- Do not provide the final answer to the user's question.
- Return a valid JSON object only.
- Do not use Markdown code fences or extra text.

Required JSON format:
{{
    "thinking": "Brief explanation of the selected sections",
    "node_list": ["node_id1", "node_id2"]
}}
"""

    # 3. Call Claude
    response = anthropic_client.messages.create(
        model=model,
        max_tokens=1500,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    # 4. Extract text from Claude's response
    response_text = "".join(
        block.text
        for block in response.content
        if block.type == "text"
    ).strip()

    if not response_text:
        raise ValueError("Claude returned an empty response.")

    # 5. Parse JSON safely
    try:
        result = json.loads(response_text)

    except json.JSONDecodeError:
        # Remove Markdown code fences if present
        cleaned = re.sub(
            r"\A```(?:json)?\s*|\s*```\Z",
            "",
            response_text,
            flags=re.IGNORECASE
        ).strip()

        try:
            result = json.loads(cleaned)

        except json.JSONDecodeError:
            # Try extracting a JSON object from surrounding text
            match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)

            if not match:
                print("Claude's raw response:")
                print(repr(response_text))
                raise ValueError(
                    "Claude did not return a valid JSON object."
                )

            try:
                result = json.loads(match.group(0))
            except json.JSONDecodeError as exc:
                print("Claude's raw response:")
                print(repr(response_text))
                raise ValueError(
                    "Could not parse Claude's JSON response."
                ) from exc

    # 6. Validate the returned structure
    if not isinstance(result, dict):
        raise ValueError("Expected a JSON object from Claude.")

    if not isinstance(result.get("node_list"), list):
        raise ValueError(
            "Claude's response must contain a node_list array."
        )

    if not isinstance(result.get("thinking", ""), str):
        result["thinking"] = str(result["thinking"])

    # Keep only node IDs that actually exist in the document tree
    valid_node_ids = set()

    def collect_ids(nodes):
        for node in nodes:
            if "node_id" in node:
                valid_node_ids.add(str(node["node_id"]))

            collect_ids(node.get("nodes", []))

    collect_ids(tree)

    result["node_list"] = [
        str(node_id)
        for node_id in result["node_list"]
        if str(node_id) in valid_node_ids
    ]

    return result


# 7. Run the query
query = "What is the course Data hub?"

print(f"🔍 Query: {query}\n")

result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Explanation:")
print(result.get("thinking", "N/A"))

print("\n🎯 Selected Node IDs:")
print(result.get("node_list", []))


🔍 Query: What is the course Data hub?

🧠 LLM Explanation:
The user is asking about 'Course Data Hub'. The most relevant nodes are the ones directly describing what Course Data Hub is and its related content. Node 0002 is the parent section for Course Data Hub, and node 0003 is the intro that directly explains what Course Data Hub/Course Master is. Additional child nodes (0004-0011) provide further details about the module.

🎯 Selected Node IDs:
['0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011']


In [30]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [35]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "claude-sonnet-4-6") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = anthropic_client.messages.create(
            model=model,
            max_tokens=1500,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )
    
    return response.choices[0].message.content

In [36]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [37]:

# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What are the course serial?",
    tree=pageindex_tree
)

🔍 Query: What are the course serial?

🧠 Reasoning: The user is asking about 'course serial'. The document tree has a section specifically titled 'Course Serial - Introduction' (node 0012) with its intro (node 0013), and child nodes covering creation s...
🎯 Retrieved node IDs: ['0012', '0013', '0014', '0015', '0016', '0017', '0018']
📄 Sections found: ['Course Serial - Introduction', 'Course Serial - Introduction (intro)', 'Getting Started with Course Serial Creation', 'Getting Started with Course Serial Creation (intro)', 'Step 4B – Confirm Updated Details', 'Creating the Other Assessment Marking Scheme', 'Finalizing and Submitting for Approval']


AttributeError: 'Message' object has no attribute 'choices'

In [38]:
import json
from anthropic import Anthropic

# Initialize Anthropic client
anthropic_client = Anthropic()


# ── Step 1: Find nodes by their IDs ──────────────────────────────────────────

def find_nodes_by_ids(tree: list, node_ids: list) -> list:
    """Recursively retrieve tree nodes matching the requested IDs."""

    found_nodes = []
    target_ids = {str(node_id) for node_id in node_ids}

    def traverse(nodes):
        for node in nodes:
            if not isinstance(node, dict):
                continue

            node_id = node.get("node_id", node.get("id"))

            if node_id is not None and str(node_id) in target_ids:
                found_nodes.append(node)

            children = node.get("nodes", node.get("children", []))

            if isinstance(children, list):
                traverse(children)

    traverse(tree)
    return found_nodes


# ── Step 2: Search the PageIndex tree ────────────────────────────────────────

def llm_tree_search(query: str, tree: list,
                    model: str = "claude-sonnet-4-6") -> dict:
    """Use Claude to select the most relevant nodes from the tree."""

    tree_json = json.dumps(tree, ensure_ascii=False, default=str)

    prompt = f"""
You are a document retrieval assistant.

Find the tree nodes most relevant to answering the user's question.

Question:
{query}

Document tree:
{tree_json}

Return ONLY valid JSON in this format:
{{
    "thinking": "Brief explanation of the selection",
    "node_list": ["node_id_1", "node_id_2"]
}}

Rules:
- Select relevant nodes only.
- Use node IDs exactly as provided in the tree.
- Return an empty node_list if no relevant nodes exist.
- Do not invent node IDs.
"""

    response = anthropic_client.messages.create(
        model=model,
        max_tokens=1500,
        messages=[{"role": "user", "content": prompt}]
    )

    result_text = "".join(
        block.text
        for block in response.content
        if block.type == "text"
    ).strip()

    # Handle JSON enclosed in Markdown fences
    if result_text.startswith("```"):
        result_text = result_text.split("\n", 1)[1]
        result_text = result_text.rsplit("```", 1)[0].strip()

    result = json.loads(result_text)

    if not isinstance(result.get("node_list"), list):
        raise ValueError("Tree search did not return a valid node_list.")

    return result


# ── Step 3: Generate an answer from retrieved nodes ──────────────────────────

def generate_answer(
    query: str,
    nodes: list,
    model: str = "claude-sonnet-4-6"
) -> str:
    """Generate an answer grounded in the retrieved document sections."""

    if not nodes:
        return "No relevant sections found in the document."

    context_parts = []

    for node in nodes:
        title = node.get("title", "Untitled section")
        page = node.get("page_index", node.get("page", "Unknown"))
        content = node.get(
            "text",
            node.get("content", node.get("summary", "Content unavailable."))
        )

        context_parts.append(
            f"[Section: {title} | Page: {page}]\n{content}"
        )

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""
You are an expert document analyst.

Answer the question using ONLY the supplied document context.

Instructions:
- Be concise and precise.
- Cite the relevant section title and page number.
- Do not invent facts or course serial numbers.
- If the answer is not present in the context, clearly say so.

Question:
{query}

Document context:
{context}

Answer:
"""

    response = anthropic_client.messages.create(
        model=model,
        max_tokens=1500,
        messages=[{"role": "user", "content": prompt}]
    )

    # Correct Anthropic SDK response access
    return "\n".join(
        block.text
        for block in response.content
        if block.type == "text"
    ).strip()


# ── Step 4: Complete Vectorless RAG pipeline ─────────────────────────────────

def vectorless_rag(
    query: str,
    tree: list,
    verbose: bool = True,
    model: str = "claude-sonnet-4-6"
) -> str:
    """
    Step 1: Search the document tree.
    Step 2: Retrieve the selected nodes.
    Step 3: Generate a grounded answer.
    """

    if verbose:
        print("=" * 60)
        print(f"Question: {query}")
        print("=" * 60)

    # Search tree
    search_result = llm_tree_search(query, tree, model=model)
    node_ids = search_result.get("node_list", [])

    if verbose:
        print("\nTree search reasoning:")
        print(search_result.get("thinking", "No reasoning returned"))
        print("\nRetrieved node IDs:", node_ids)

    # Retrieve matching sections
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print("\nRetrieved sections:")
        for node in nodes:
            print("-", node.get("title", "Untitled section"))

        if node_ids and not nodes:
            print(
                "Warning: No matching nodes found. "
                "Check your tree's actual node ID and child field names."
            )

    # Generate grounded answer
    answer = generate_answer(query, nodes, model=model)

    if verbose:
        print("\nAnswer:")
        print(answer)
        print("=" * 60)

    return answer

In [39]:
answer = vectorless_rag(
    query="What are the course serial numbers?",
    tree=pageindex_tree
)

print("\nFinal Answer:", answer)

Question: What are the course serial numbers?

Tree search reasoning:
The user is asking about 'course serial numbers'. Looking at the document tree, the Course Serial section introduces the concept with examples like BOM-101, BOM-102, or AMM-57. The introduction to Course Serial (node 0013) explicitly mentions these examples. The getting started section (node 0015) also covers how to enter the Serial Name when creating a course serial. These nodes are most relevant to understanding what course serial numbers are.

Retrieved node IDs: ['0013', '0015']

Retrieved sections:
- Course Serial - Introduction (intro)
- Getting Started with Course Serial Creation (intro)

Answer:
## Course Serial Numbers

Based on the document context, the course serial numbers mentioned as examples are:

- **BOM-101**
- **BOM-102**
- **AMM-57**

These are cited in **Section: Course Serial - Introduction (Page: Unknown)** as examples of specific instances or batches of a course conducted at a particular time.


In [40]:
FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings
"""

print("✅ Expert rules defined")
print("   These get injected into the retrieval prompt at query time.")

✅ Expert rules defined
   These get injected into the retrieval prompt at query time.
